# Working with Spatial Data in Python — Vector Data
**John Fay · Fall 2026**  


## Highlights

- Vector data represent spatial features as **geometries**: points, lines, and polygons.
  - *Point* = single coordinate pair
  - *Line* = sequence of coordinate pairs
  - *Polygon* = sequence of coordinate pairs, with the last overlapping the first.
- Geometries are stored as coordinate pairs in their own column in a **GeoDataFrame**.
- Geometries and spatial dataframes are also associated with a **coordinate reference system** (crs)
  - The crs defines origin and axes to which the coordinate pairs are referenced.
  - A crs can be geographic and use angular units (e.g. latitude/longitude) ;\
    Or it can be projected and use linear units (e.g. UTM, State Plane)

By the end, you should be able to create points from a table, inspect their CRS, display an interactive map, and attach county attributes using a spatial join.

## 1. Prepare your environment and workspace
In a Miniforge terminal, create an environment once:
```bash
conda create -n env700-vector -c conda-forge python=3.12 geopandas pandas matplotlib folium mapclassify jupyterlab ipykernel
```
Open this notebook in VS Code and select **Python (env700-vector)** using **Select Kernel**. 


In [ ]:
#Packages for working with data files/remote data
from pathlib import Path
from urllib.request import urlopen
import platform

#Packages for working with data
import pandas as pd
import geopandas as gpd

#Packages for viewing data
import matplotlib.pyplot as plt
import folium
from IPython.display import display

### Locate the project folders

In [ ]:
#Paths
PROJECT_ROOT = Path.cwd().parent
raw_folder = PROJECT_ROOT / "Data" / "raw"
processed_folder = PROJECT_ROOT / "Data" / "processed"
print("Project folder:", PROJECT_ROOT)
print("Raw data folder:", raw_folder)

### Optional: locate Box Drive

In [ ]:
#Determine if using a PC, Mac, or Unix
system = platform.system()
if system == "Darwin":
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"
elif system == "Windows":
    box_path = Path.home() / "Box"
else:
    box_path = None

#Report results
if box_path is None:
    print("No Box Drive path configured for this operating system.")
else:
    print("Candidate Box path:", box_path)
    print("Folder exists:", box_path.exists())

## 2. Reading in spatial Data

### 2.1 Converting a dataframe with coordinate data into a GeoDataFrame
If your dataframe has coordinate data, i.e., columns for X and Y coordinates, you can create a column of geospatial point objects, called a **GeoSeries**, from these coordiate column using GeoPandas' `points_from_xy()` function. Once created, you can then create a GeoPandas **GeoDataFrame** from your original dataframe along with this GeoSeries object.

Here we read in the `NWIS_SiteInfo_NE_RAW.csv` file, which has an "x" (`dec_long_va`) and a "y" (`dec_lat_va`) column. We then create the GeoSeries object from these coordinate columns, and from that construct our GeoDataFrame via GeoPandas' `GeoDataFrame()` function.

Note we have to also provide the **coordinate reference system (crs)**. We do so by providing its **EPSG code**; our data has coordinates that are latitude/longitude, so we use the EPSG code of **4326** which represents the World Geodetic Survey of 1984. (See <https://epsg.io/>), often used for data with latitude/longitude coordinates. 

### 2.1 Read the NWIS site table with Pandas. 

In [ ]:
#Set the filename
NWIS_filename = raw_folder / "NWIS_SiteInfo_NE_RAW.csv"

#Read the file into a dataframe
NWIS_df = pd.read_csv(NWIS_filename, dtype={"site_no": "string"})

#Display
NWIS_df.head()

In [ ]:
#Create the GeoSeries object from the coordinate columns
the_geoseries = gpd.points_from_xy(
        x= NWIS_df["dec_long_va"], 
        y= NWIS_df["dec_lat_va"]
    )

#Construct a geodataframe from the dataframe and geoseries, specifying the crs
NWIS_gdf = gpd.GeoDataFrame(
    data = NWIS_df,
    geometry= the_geoseries,
    crs="EPSG:4326"
)

#Display - notice the geometry column 
NWIS_gdf.head()

### 2.3 Viewing GeoDataFrames

In [ ]:
#View as a simple static map
simple_map = NWIS_gdf.plot(markersize=3)
simple_map;

In [ ]:
#View as an interactive map (requires internet connection)
NWIS_gdf.explore(tiles="OpenStreetMap")

### 2.3 Reading GeoJSON files directly from the internet

GeoJSON is a text file format that stores all the information of a geospatial feature class: geometries, attributes, and coordinate reference system information. The are text based equivalents of Shapefiles, but since they have all the data in a single file, they are a bit easier to work with. In fact, a web site can host these data on-line and we can read them directly into a spatial dataframe using `st_read()`.

A quick web search for "US Counties GeoJSON" reveals many sources of that dataset. The one that pop up in the result for me is: <https://eric.clst.org/tech/usgeojson/> - an individual who has decided to host many datasets on his personal web site.

The link for US Counties at a 1:5m scale is: [`https://eric.clst.org/assets/wiki/uploads/Stuff/gz_2010_us_050_00_5m.json`](https://eric.clst.org/assets/wiki/uploads/Stuff/gz_2010_us_050_00_5m.json).

We'll fetch that, reading its contents into a spatial dataframe, filtering for just counties in Nebraska, whose state FIPS code is "31".

In [ ]:
#Set the URL
counties_url = "https://eric.clst.org/assets/wiki/uploads/Stuff/gz_2010_us_050_00_5m.json"

#Fetch as a geodataframe
counties_gdf = gpd.read_file(counties_url,encoding="ISO-8859-1")

#Filter for Nebraska (FIPS = 31)
counties_gdf = counties_gdf.loc[counties_gdf['STATE']=='31'].reset_index()

#Display as table
counties_gdf.head()

In [ ]:
#Display as map
counties_gdf.plot();

## 3. Spatial Operations

### 3.1 Spatial Join
A spatial join assigns values in one dataset based on its overlap with a second spatial dataset. For example, if we wanted to tag each of our NWIS points with the county in which it falls, we would spatially join the county data to the points. This is done via the GeoPandas `sjoin()` function.

In [ ]:
#Spatiall join the county attributes to the NWIS points
NWIS_joined = gpd.sjoin(
    left_df = NWIS_gdf,
    right_df = counties_gdf,
    predicate='intersects'
)

#Display info on the result - note the newly added columns
NWIS_joined.info()

In [ ]:
#Plot the result
base_map = counties_gdf.plot(color='lightgray',edgecolor='gray',linewidth=0.5,figsize=(10,6))
NWIS_joined.plot(ax=base_map,markersize=NWIS_joined['CENSUSAREA'] / 100);

### 3.2 Spatial Filtering
We can also select features in one spatial dataframe based on its spatial relationship to features in another dataframe, e.g., features in one layer that overlap features in another. The process is analogous to creating a Boolean mask and then subsetting records where that mask is true, except that the Boolean mask is created via a spatial overlay. 

To demonstrate this, we'll extract a subset of NWIS points that overlap a specific watershed, the Platte River [HUC4](https://nas.er.usgs.gov/hucs.aspx), which we can fetch from <https://data.openwaterfoundation.org/country/us/usgs/huc-basins/>. Specifically, the URL for the Platte River HUC4 (1020) data is: https://data.openwaterfoundation.org/country/us/usgs/huc-basins/huc4-basin-1020-platte.geojson

In [ ]:
#Fetch the North Platte HUC4 
huc4_url = 'https://data.openwaterfoundation.org/country/us/usgs/huc-basins/huc4-basin-1020-platte.geojson'
huc4_gdf = gpd.read_file(huc4_url) 

#Display the HUC4 with the NWIS points
base_map = counties_gdf.plot(color='lightgray',edgecolor='gray',linewidth=0.5,figsize=(10,6))
huc4_gdf.plot(ax=base_map,color='#98AFC7',linewidth=0.3,edgecolor='blue')
NWIS_joined.plot(ax=base_map,markersize=9,color='darkred');

Now, we'll filter the NWIS sites, selecting only those that intersect the features in the Platte River HUC4.

In [ ]:
#Create the Boolean mask: True for NWIS points intersecting the Platte River HUC4; false for all others
HUC4_mask = NWIS_gdf.intersects(huc4_gdf.geometry.union_all()) 

#Filter the NWIS sites that intersect the Platte River HUC4
NWIS_selected = NWIS_gdf[HUC4_mask]

#report how many we selected
print(f'Of the {NWIS_gdf.shape[0]} NWIS points, {NWIS_selected.shape[0]} points fall within the Platte R. HUC4')

> 🔥See the *Geometric Confirmation* section in the GeoPandas cheat sheet for other ways to spatially select features.

In [ ]:
#Display
base_map = counties_gdf.plot(color='lightgray',edgecolor='gray',linewidth=0.5,figsize=(10,6)) # County base map
huc4_gdf.plot(ax=base_map,color='#98AFC7',linewidth=0.3,edgecolor='blue')                   # Plot the HUC4
NWIS_joined.plot(ax=base_map,markersize=3,color='darkred')                                    # Plot all the points in darkred
NWIS_selected.plot(ax=base_map,markersize=9,color='red');                                     # Plot the subset points in red

### 3.3 Clipping Features

In addition to spatially selecting features, we can clip features in one spatial dataframe using features in another. Here we'll clip the counties to areas that intersect the Platte River HUC4.

In [ ]:
#Clip the counties to the area that falls within the HUC4 polygons
counties_clip = gpd.overlay(
    df1 = counties_gdf, # The layer we want to clip (cookie dough)
    df2 = huc4_gdf,     # The layer used to clip (the cookie cutter)
    how='intersection'  # How we want to clip the data
)

In [ ]:
# View the result
counties_clip.plot();

### 3.4 Spatial Aggregation 
If you want to combine spatial features

In [ ]:
# Initialize the column values
counties_HUC = counties_gdf.copy()
counties_HUC["InHUC"] = "OUT"

# Create Boolean mask for counties intersecting HUC4
counties_intersect_mask = counties_HUC.intersects(
    huc4_gdf.geometry.union_all()
)

# Update records where mask is True
counties_HUC.loc[counties_intersect_mask, "InHUC"] = "IN"

# Show results
base_map = counties_HUC.plot(column="InHUC",legend=True,figsize=(10,6))
huc4_gdf.plot(ax=base_map,facecolor='none',edgecolor='black');

### 3.5 Geometry manipulations

We can alter the geometries in a spatial dataframe. For example, we can buffer features or reduce polygons to their centroids. In doing these calculations, however, we want to ensure our data use a projected coordinate reference system, i.e., one where the x's and y's are in linear units (meters or feet) and not angular units (latitude/longitude).

Without going into gory detail, you'll want to transform your data to a UTM coordinate system. Because we are in Nebraska, we'll use UTM Zone 14N, whose EPSG code is **32614**.

In [ ]:
#Transform each of the geodataframes to a projected coordinate system (EPSG: 32614)
NWIS_utm = NWIS_gdf.to_crs(32614)
counties_utm = counties_gdf.to_crs(32614)
huc4_utm = huc4_gdf.to_crs(32614)

Now that the geometry are in meters, we sensibly measure distances between objects and buffer.

Here are some examples:

In [ ]:
#Extract the centroids of the selected features and show them
county_centroids = counties_utm.copy()
county_centroids['geometry'] = county_centroids.geometry.centroid

#Plot
base_map = counties_utm.plot(color='lightgray',edgecolor='gray',linewidth=0.5,figsize=(10,6))
county_centroids.plot(ax=base_map,markersize=4,color='darkred');

In [ ]:
#Buffer the centroids outward 5km and add them to our map
county_centroids_out5km = county_centroids.copy()
county_centroids_out5km['geometry'] = county_centroids_out5km.buffer(5000)

#Plot
base_map = counties_utm.plot(color='lightgray',edgecolor='gray',linewidth=0.5,figsize=(10,6))
county_centroids.plot(ax=base_map,markersize=4,color='darkred');
county_centroids_out5km.plot(ax=base_map,markersize=9,facecolor='none',edgecolor='blue',alpha=0.5);


In [ ]:
#Buffer the counties inward 5km
counties_in5km = counties_utm.copy()
counties_in5km['geometry'] = counties_in5km.buffer(-5000)

#Plot
base_map = counties_utm.plot(color='lightgray',edgecolor='gray',linewidth=0.5,figsize=(10,6))
counties_in5km.plot(ax=base_map,facecolor='green',edgecolor='none',alpha=0.5);

In [ ]:
#Construct a convex hull around the centroids
county_centroids_chull = county_centroids.copy()
county_centroids_chull['geometry'] = county_centroids_chull.geometry.union_all().convex_hull

#Plot the convex hull
base_map = counties_utm.plot(color='lightgray',edgecolor='gray',linewidth=0.5,figsize=(10,6))
county_centroids.plot(ax=base_map,markersize=4,color='darkred')
county_centroids_chull.plot(ax=base_map,facecolor='none',edgecolor='red',alpha=0.5);

In [ ]:

#Plot the convex hull
base_map = counties_utm.plot(color='lightgray',edgecolor='gray',linewidth=0.5,figsize=(10,6))
county_centroids_chull.plot(ax=base_map,facecolor='none',edgecolor='red',alpha=0.5);

In [ ]:


#Combine the centroids into one feature and construct a convex hull around them
county_centroids_chull <- county_centroids %>% 
  st_union() %>% 
  st_convex_hull()
myMap <- myMap + geom_sf(data = county_centroids_chull, color = 'red', fill=NA)
myMap

#Combine the centroids into one feature and draw voronoi polygons
county_centroids_voronoi <- county_centroids %>% 
  st_union() %>% 
  st_voronoi()
myMap <- myMap + geom_sf(data = county_centroids_voronoi, color = 'purple', fill=NA)
myMap